# IEEE fraud model training

Reads the processed train, validation, and test parquet files and trains an XGBoost model. The split already happened in `01_data_transformation.ipynb`.

Tracking goes to the Docker MLflow server (`http://localhost:5000`). That server proxies artifacts into the MinIO bucket `s3://mlflow/` (`http://localhost:9000`).

In [1]:
from pathlib import Path
import os

import mlflow
import pandas as pd

from src.run_time_configuration import BaseConfigParams
from src.model_training.training import ModelTrainingWorkflow

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "runs_entrypoints":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TARGET_COLUMN = "is_fraud"
SEED = 42
TOP_N_FEATURES = 40
DROP_FROM_FEATURES = ["transaction_id", "seconds_from_reference", TARGET_COLUMN]


In [2]:

def _load_env(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}")
    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())


_load_env(PROJECT_ROOT / ".env")

MLFLOW_TRACKING_URI = "http://localhost:5000"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed dir: {PROCESSED_DIR}")
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

Project root: /home/martinezyamamotoarthur/modeling_fraud_system
Processed dir: /home/martinezyamamotoarthur/modeling_fraud_system/data/processed
MLflow tracking URI: http://localhost:5000


In [3]:
df_train = pd.read_parquet(PROCESSED_DIR / "train.parquet")
df_validation = pd.read_parquet(PROCESSED_DIR / "validation.parquet")
df_test = pd.read_parquet(PROCESSED_DIR / "test.parquet")

for name, frame in {
    "train": df_train,
    "validation": df_validation,
    "test": df_test,
}.items():
    print(name, frame.shape, "pos_rate", round(float(frame[TARGET_COLUMN].mean()), 4))

train (56224, 458) pos_rate 0.0349
validation (14056, 458) pos_rate 0.0349
test (29720, 458) pos_rate 0.0352


In [4]:
def features_and_target(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    y = df[TARGET_COLUMN]
    x = df.drop(columns=[column for column in DROP_FROM_FEATURES if column in df.columns])
    return x, y


runtime_params = BaseConfigParams(
    experiment_name="ieee_fraud",
    mlflow_run_name="ieee_xgb",
    training_start_date="2017-12-01",
    training_end_date="2018-03-31",
    test_start_date="2018-04-01",
    test_end_date="2018-06-30",
)

workflow = ModelTrainingWorkflow(runtime_params)
workflow.df_train = df_train
workflow.df_validation = df_validation
workflow.df_test = df_test
workflow.x_train, workflow.y_train = features_and_target(df_train)
workflow.x_validation, workflow.y_validation = features_and_target(df_validation)
workflow.x_test, workflow.y_test = features_and_target(df_test)
workflow.generate_preprocessed_data(numeric_only=True)

print("train", workflow.x_train_preproc.shape, "pos_rate", float(workflow.y_train.mean()))
print("valid", workflow.x_validation_preproc.shape, "pos_rate", float(workflow.y_validation.mean()))
print("test ", workflow.x_test_preproc.shape, "pos_rate", float(workflow.y_test.mean()))
print("numeric features", len(workflow.x_train_preproc.columns))

2026-09-27 10:55:41,426 - src.model_training.training - INFO - Initialized the SolvencyNewCustomersModelTraining workflow.
2026-09-27 10:55:41,434 - src.model_training.training - INFO - Generating preprocessed data by copying original data.
2026-09-27 10:55:41,718 - src.model_training.training - INFO - Filtering to numeric columns: 409 out of 455 total columns
2026-09-27 10:55:41,719 - src.model_training.training - INFO - Selected numeric columns: ['amount_usd', 'card1', 'card2', 'card_issue_country', 'card5', 'billing_region', 'billing_country', 'dist1', 'dist2', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'transaction_time_of_day', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'V29', 'V30', 'V31', 'V32', 'V33', '

train (56224, 409) pos_rate 0.03491391576550939
valid (14056, 409) pos_rate 0.03493170176437109
test  (29720, 409) pos_rate 0.03519515477792732
numeric features 409


In [5]:
results = workflow.train_model(
    hyperparameters={
        "n_estimators": 200,
        "max_depth": 5,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    model_type="xgboost",
    log_into_mlflow=True,
    show_plots=False,
    verbose=False,
    seed=SEED,
    run_name="ieee_xgb_01",
    top_n_features=TOP_N_FEATURES,
)

metrics_table = pd.DataFrame(
    {
        "train": results["train_metrics"],
        "validation": results["val_metrics"],
        "test": results["test_metrics"],
    }
).loc[["accuracy", "precision", "recall", "f1", "auc", "pr_auc", "ks_statistic"]]

run_id = workflow.run_time_config.mlflow_run_id
client = mlflow.tracking.MlflowClient()
run = client.get_run(run_id)

print("threshold", round(results["threshold"], 4))
print("mlflow run id", run_id)
print("tracking uri", mlflow.get_tracking_uri())
print("artifact uri", run.info.artifact_uri)
print("logged artifacts", [a.path for a in client.list_artifacts(run_id)])
display(results["top_features"])
metrics_table.round(4)

2026-09-27 10:55:41,967 - src.model_training.training - INFO - Starting model training without hyperparameter tuning using xgboost.
2026-09-27 10:55:42,041 - src.model_training.training - INFO - Training xgboost model with fixed hyperparameters.
/home/martinezyamamotoarthur/modeling_fraud_system/.venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [10:55:42] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
2026-09-27 10:55:49,368 - src.model_training.training - INFO - Calculated optimal threshold (F1-based) on validation set: 0.1855
2026-09-27 10:55:49,518 - src.model_training.evaluation - INFO - Training Metrics - AUC: 0.9282, PR-AUC: 0.6819, KS: 0.6997
2026-09-27 10:55:49,537 - src.model_training.evaluation - INFO - Validation Metrics - AUC: 0.8886, PR-AUC: 0.5657, KS: 0.6249
2026-09-27 10:55:49,568 - src.model_training.evaluation - INFO - Test Metrics - AUC: 0.8699, P

🏃 View run ieee_xgb_01 at: http://localhost:5000/#/experiments/52/runs/7f326500ad9d47928486bac7187d27ab
🧪 View experiment at: http://localhost:5000/#/experiments/52
threshold 0.1855
mlflow run id 7f326500ad9d47928486bac7187d27ab
tracking uri http://localhost:5000
artifact uri mlflow-artifacts:/52/7f326500ad9d47928486bac7187d27ab/artifacts
logged artifacts ['feature_importance', 'plots']


,rank,feature,importance
0,1,V258,568.911560
1,2,V243,480.497498
2,3,V201,315.459930
3,4,V268,187.390961
4,5,C8,139.917740
5,6,C4,132.020554
6,7,V317,131.371948
7,8,V294,119.200882
8,9,V70,80.121986
9,10,C10,75.978096


,train,validation,test
accuracy,0.979013,0.973677,0.96716
precision,0.758757,0.663073,0.545692
recall,0.584819,0.501018,0.399618
f1,0.660529,0.570766,0.461369
auc,0.92822,0.888613,0.869893
pr_auc,0.681882,0.565742,0.451207
ks_statistic,0.699681,0.624933,0.587314
